In [ ]:
# Import libraries. You may or may not use all of these.
!pip install -q git+https://github.com/tensorflow/docs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
  # %tensorflow_version only exists in Colab.
  %tensorflow_version 2.x
except Exception:
  pass
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

import tensorflow_docs as tfdocs
import tensorflow_docs.plots
import tensorflow_docs.modeling

In [ ]:
# Import data
!wget https://cdn.freecodecamp.org/project-data/health-costs/insurance.csv
dataset = pd.read_csv('insurance.csv')
dataset.tail()

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Normalize numeric features
from sklearn.preprocessing import StandardScaler

# Convert categorical data to numbers
CATEGORICAL_COLUMNS = ['sex', 'smoker', 'region']
NUMERIC_COLUMNS = ['age', 'bmi', 'children', 'expenses']

feature_columns = []
for feature_name in CATEGORICAL_COLUMNS:
  vocabulary = dataset[feature_name].unique()
  feature_columns.append(tf.feature_column.categorical_column_with_vocabulary_list(feature_name, vocabulary))

for feature_name in NUMERIC_COLUMNS:
  feature_columns.append(tf.feature_column.numeric_column(feature_name, dtype=tf.float32))

# Calculate the split index for 80% training and 20% testing
split_index = int(len(dataset) * 0.8)

# Split the dataset into training and testing sets
train_dataset = dataset[:split_index]
test_dataset = dataset[split_index:]

# pop off the "expenses" column from these datasets
train_labels = train_dataset['expenses']
test_labels = test_dataset['expenses']

scaler = StandardScaler()
train_dataset[NUMERIC_COLUMNS] = scaler.fit_transform(train_dataset[NUMERIC_COLUMNS])
test_dataset[NUMERIC_COLUMNS] = scaler.transform(test_dataset[NUMERIC_COLUMNS])

# Convert categorical features to one-hot encoding
train_dataset = pd.get_dummies(train_dataset, columns=CATEGORICAL_COLUMNS)
test_dataset = pd.get_dummies(test_dataset, columns=CATEGORICAL_COLUMNS)

# Ensure both datasets have the same columns
test_dataset = test_dataset.reindex(columns=train_dataset.columns, fill_value=0)

# Build the Keras model
model = Sequential([
    Dense(64, activation='relu', input_shape=(train_dataset.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1)  # Output layer for regression
])

# Compile the model
model.compile(optimizer='adam', loss='mse', metrics=['mae', 'mse'])

# Train the model
history = model.fit(
    train_dataset, train_labels,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    verbose=1
)

In [ ]:
# RUN THIS CELL TO TEST YOUR MODEL. DO NOT MODIFY CONTENTS.
# Test model by checking how well the model generalizes using the test set.
loss, mae, mse = model.evaluate(test_dataset, test_labels, verbose=2)

print("Testing set Mean Abs Error: {:5.2f} expenses".format(mae))

if mae < 3500:
  print("You passed the challenge. Great job!")
else:
  print("The Mean Abs Error must be less than 3500. Keep trying.")

# Plot predictions.
test_predictions = model.predict(test_dataset).flatten()

a = plt.axes(aspect='equal')
plt.scatter(test_labels, test_predictions)
plt.xlabel('True values (expenses)')
plt.ylabel('Predictions (expenses)')
lims = [0, 50000]
plt.xlim(lims)
plt.ylim(lims)
_ = plt.plot(lims,lims)
